# Gold Standard 정합성 평가 — LLM × 임베딩 비교

`data2` 의 gold standard 50건에 대해:
1. **Stage A 추출**을 여러 LLM(`config.CLAIM_MODELS`)으로 수행하고
2. 각 폴더 `*_gold.json` 의 **gold_claims** 와 정합성을 확인합니다.
3. 정합성은 **임베딩 유사도**(코사인)로 측정하고,
4. **추출 LLM × 평가 임베딩** 조합을 비교합니다.
   - LLM: `config.CLAIM_MODELS` (qwen2.5-7b, exaone-3.5-7.8b, gemma-2-9b-it …)
   - 평가 임베딩: `BAAI/bge-m3`, `jhgan/ko-sroberta-multitask`, `BM-K/KoSimCSE-roberta-multitask`

> 지표: 예측 claim ↔ gold claim 을 임베딩으로 대조해 **soft/hard P·R·F1** 계산.
> 추출 LLM 은 claim 을 만들고, 평가 임베딩은 그 claim 이 gold 와 얼마나 맞는지 재는 데 씁니다(서로 별개).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path

from safety_pipeline import config
from safety_pipeline.dataio import load_samples_from_data_dir, load_all_claim_sets
from safety_pipeline.stage_a import run_stage_a
from safety_pipeline.consensus import ClaimEmbedder
from safety_pipeline.gold_eval import load_gold_by_sample, evaluate_overall, evaluate_by_type, match_detail_table

In [ ]:
# ---- 평가 설정 ----
DATA2_DIR = "../data2"
THRESHOLD = 0.7          # hard 매칭 임계값(코사인 유사도)
EVAL_DEVICE = "cpu"      # 임베딩 추론 장치 ("cuda" 로 바꾸면 GPU)
SAMPLE_LIMIT = 5         # 빠른 검증용 앞 N건만. 전체(50건) 실행은 None 으로 변경
VERBOSE_STAGE_A = True   # Stage A 진행 로그 출력

# 정합성 측정용 임베딩 모델 (추출 LLM 과 별개)
EVAL_EMBEDDING_MODELS = {
    "bge-m3": {"model_id": "BAAI/bge-m3"},
    "ko-sroberta-multitask": {"model_id": "jhgan/ko-sroberta-multitask"},
    "kosimcse-roberta-multitask": {"model_id": "BM-K/KoSimCSE-roberta-multitask"},
}

print("비교할 추출 LLM:", list(config.CLAIM_MODELS.keys()))

## 1. Stage A 추출 (여러 LLM × data2 50건)

`config.CLAIM_MODELS` 의 각 LLM 으로 claim 을 추출·합의합니다. LLM 은 **순차 로드/언로드**되며
VRAM 에는 한 번에 한 모델만 올라갑니다. (LLM 3개 × 50건이면 수십 분 소요될 수 있습니다.)

> gemma-2 는 HF 라이선스 동의 + 토큰 로그인이 필요합니다. EXAONE 은 `trust_remote_code=True` 로 로드합니다.

In [ ]:
samples = load_samples_from_data_dir(DATA2_DIR)
if SAMPLE_LIMIT:
    samples = samples[:SAMPLE_LIMIT]
print(f"실행할 샘플 수: {len(samples)}  (SAMPLE_LIMIT={SAMPLE_LIMIT})")

# 이미 추출을 돌렸다면 아래로 재추출을 건너뛸 수 있습니다:
# claim_sets_by_combo = load_all_claim_sets(config.DEFAULT_STAGE_A_DIR)

claim_sets_by_combo = run_stage_a(samples, verbose=VERBOSE_STAGE_A)   # 모든 CLAIM_MODELS 실행
for (llm, cons_emb), cs in claim_sets_by_combo.items():
    print(f"[{llm} × {cons_emb}] claim 집합 {len(cs)}개, 총 claim {sum(len(c.claims) for c in cs)}개")

## 2. Gold 정합성 — 추출 LLM × 평가 임베딩 그리드

각 (LLM, 평가 임베딩) 조합에 대해 추출 claim ↔ gold_claims 정합성 지표를 계산합니다.

In [ ]:
gold_map = load_gold_by_sample(DATA2_DIR)
print("gold 로드:", len(gold_map), "샘플")

overall_rows = []
per_sample_frames = {}   # {(llm, eval_emb): DataFrame}
per_type_frames = {}     # {(llm, eval_emb): DataFrame}

for emb_name, emb_cfg in EVAL_EMBEDDING_MODELS.items():
    print(f"\n===== 평가 임베딩: {emb_name} ({emb_cfg['model_id']}) =====")
    emb = ClaimEmbedder(emb_name, {**emb_cfg, "device": EVAL_DEVICE})
    for (llm, cons_emb), cs_list in claim_sets_by_combo.items():
        df = pd.DataFrame(evaluate_overall(cs_list, gold_map, emb, threshold=THRESHOLD))
        per_sample_frames[(llm, emb_name)] = df
        per_type_frames[(llm, emb_name)] = pd.DataFrame(evaluate_by_type(cs_list, gold_map, emb, threshold=THRESHOLD))
        overall_rows.append({
            "claim_llm": llm,
            "eval_embedding": emb_name,
            "mean_soft_precision": round(df["soft_precision"].mean(), 4),
            "mean_soft_recall":    round(df["soft_recall"].mean(), 4),
            "mean_soft_f1":        round(df["soft_f1"].mean(), 4),
            f"mean_precision@{THRESHOLD}": round(df["precision"].mean(), 4),
            f"mean_recall@{THRESHOLD}":    round(df["recall"].mean(), 4),
            f"mean_f1@{THRESHOLD}":        round(df["f1"].mean(), 4),
            "total_pred": int(df["n_pred"].sum()),
            "total_gold": int(df["n_gold"].sum()),
        })
    emb.unload()

comparison = pd.DataFrame(overall_rows).sort_values(["claim_llm", "mean_soft_f1"], ascending=[True, False])
print("\n=== LLM × 평가임베딩 비교 (샘플 평균) ===")
display(comparison)

## 3. 유형별 정합성 (LLM 비교, 평가 임베딩 고정)

평가 임베딩 하나를 고정하고, claim 유형별 soft F1 을 LLM 간 비교합니다.

In [ ]:
FIXED_EMB = list(EVAL_EMBEDDING_MODELS)[0]   # 예: bge-m3

rows = []
for (llm, en), bt in per_type_frames.items():
    if en != FIXED_EMB or bt.empty:
        continue
    g = bt.groupby("claim_type")["soft_f1"].mean()
    for t, v in g.items():
        rows.append({"claim_type": t, "claim_llm": llm, "soft_f1": round(float(v), 4)})

per_type = pd.DataFrame(rows).pivot(index="claim_type", columns="claim_llm", values="soft_f1")
per_type = per_type.reindex([t for t in config.CLAIM_TYPES if t in per_type.index])
print(f"=== 유형별 soft F1 (평가 임베딩={FIXED_EMB}, 행=유형, 열=LLM) ===")
display(per_type)

## 4. 상세 매칭 (한 LLM/샘플) — 무엇이 뽑혔고 gold 와 얼마나 맞는지

`side=pred` 는 LLM 이 뽑은 claim(정밀도), `side=gold` 는 정답 claim(재현율) 관점입니다.

In [ ]:
DETAIL_LLM = list(config.CLAIM_MODELS)[0]      # 볼 LLM
DETAIL_EMB = list(EVAL_EMBEDDING_MODELS)[0]     # 상세 매칭에 쓸 임베딩

cs_list = [v for k, v in claim_sets_by_combo.items() if k[0] == DETAIL_LLM][0]
emb = ClaimEmbedder(DETAIL_EMB, {**EVAL_EMBEDDING_MODELS[DETAIL_EMB], "device": EVAL_DEVICE})
detail = pd.DataFrame(match_detail_table(cs_list, gold_map, emb, threshold=THRESHOLD))
emb.unload()

sid = cs_list[0].sample_id
print(f"LLM={DETAIL_LLM} · 평가임베딩={DETAIL_EMB}\n{sid} | {gold_map[sid]['description'][:60]}")
display(detail[detail["sample_id"] == sid][["side", "claim_type", "text", "best_match", "best_sim", "matched"]])

## 5. 결과 저장

In [ ]:
out = Path("./comparison_results/gold_eval")
out.mkdir(parents=True, exist_ok=True)

comparison.to_csv(out / "llm_embedding_comparison.csv", index=False, encoding="utf-8-sig")
per_type.to_csv(out / "per_type_soft_f1.csv", encoding="utf-8-sig")
for (llm, en), df in per_sample_frames.items():
    df.to_csv(out / f"per_sample__{llm}__{en}.csv", index=False, encoding="utf-8-sig")
detail.to_csv(out / f"match_detail__{DETAIL_LLM}__{DETAIL_EMB}.csv", index=False, encoding="utf-8-sig")

print("저장 완료:", out.resolve())
for p in sorted(out.glob("*.csv")):
    print("  -", p.name)

## 참고

- **soft P/R/F1**: 각 claim의 상대 최고 유사도 평균(임계값 없는 연속 정합도).
- **hard P/R/F1@τ**: 최고 유사도가 τ(기본 0.7) 이상이면 매칭으로 간주한 비율.
- `precision`(정밀도)=추출 중 gold에 대응되는 비율, `recall`(재현율)=gold 중 추출이 잡아낸 비율.
- **추출 LLM 을 늘리려면**: `safety_pipeline/config.py` 의 `CLAIM_MODELS` 에 항목 추가. bf16 기준 24GB VRAM 에 ~9B 까지 적재, 14B 이상은 4bit 양자화 필요.
- **KoSimCSE 주의**: `BM-K/KoSimCSE-roberta-multitask` 는 원래 CLS 풀링 모델입니다. sentence-transformers 로 로드 시 평균 풀링으로 대체될 수 있어 원 성능과 다를 수 있습니다.
- 임계값 τ, 장치, 비교 모델은 상단 설정 셀에서 바꿀 수 있습니다.